# MTSD — Detection Dataset Preparation

**MSc Dissertation — University of Malta**

This notebook converts the group-based Maltese Traffic Sign Dataset (MTSD) into a single,
clean object-detection dataset ready for the supervised training notebooks in this folder
(`YOLO12.ipynb`, `YOLO26.ipynb`, `RF-DETR.ipynb`).

**Inputs** (read-only):

- QA-verified annotations: `Datasets/MTSD/Annotations/GRP-*/Final-QA/QA-GRP*.json`
  (COCO-style, absolute-pixel `bbox = [x, y, w, h]`, 12 sign classes, per-box auxiliary
  attributes). Only these QA exports are used — raw annotator outputs are never read.
- Source images: resolved from each annotation's `source_image` field
  (`Datasets/MTSD/GRP-<n>/Images/...`).

**Outputs** (written once, overwrite-protected):

- `Datasets/MTSD/Prepared/MTSD-YOLO/` — YOLO layout (`train|valid|test/{images,labels}` +
  `data.yaml`) for YOLO12/YOLO26.
- `Datasets/MTSD/Prepared/MTSD-COCO/` — COCO layout (`train|valid|test/` with images +
  `_annotations.coco.json`, contiguous 0-based category ids) for RF-DETR.
- `Datasets/MTSD/Prepared/prep_manifest.json` — full provenance record (configuration,
  class map, split assignment counts, integrity findings).

### Methodological notes

- **Groups are distribution artefacts, not semantic units** — they exist because
  collection was partitioned for third-party annotation. The default split is therefore a
  seeded random split stratified *per group* (each group contributes proportionally to
  every split). `SPLIT_MODE = "by-group"` is available to hold entire groups out instead
  (a stronger leakage guard at the cost of confounding split with collection batch).
- **Every image is a unique capture** (no augmented duplicates exist at this stage), so an
  image-level split cannot leak augmented copies across splits.
- **Auxiliary attributes** (view angle, mounting, condition, shape) are deliberately
  dropped here: they are the target of the separate attribute-classification pipeline
  (`Scripts/MTSD-Scripts/AttributeClassification/`), not of the detectors.
- **Offline augmentation is photometric only and disabled by default.** Traffic signs are
  *not* mirror-invariant (text, arrows, and mounting geometry flip), so geometric flips
  are excluded by design; the optional section (§7) generates brightness/contrast/blur
  variants of training images only.
- **Known data gap:** GRP-3's QA file annotates 617 images but its `Images/` folder holds
  only 398 — the ~219 missing files are reported and excluded, not silently dropped.

### Contents

1. Configuration
2. Repository paths
3. Discovery and group indexing
4. Validation of image–annotation pairs
5. Train/validation/test split
6. Write the YOLO and COCO datasets
7. Optional offline augmentation (disabled by default)
8. Post-preparation integrity checks
9. Summary report


## 1. Configuration

Every switch that controls the preparation lives in this one cell.


In [ ]:
# --- Preparation configuration (edit here, nowhere else) ----------------------
REGENERATE_SPLITS = True          # False = stop if a prep_manifest.json already exists
OVERWRITE_EXISTING = False        # True = archive any existing output tree, then rebuild
SPLIT_MODE = "random"             # "random" (per-group stratified) | "by-group"
SPLIT_RATIOS = {"train": 0.80, "valid": 0.10, "test": 0.10}
BY_GROUP_ASSIGNMENT = {"GRP-1": "train", "GRP-2": "valid", "GRP-3": "test"}
RANDOM_SEED = 42

ENABLE_OFFLINE_AUGMENTATION = False   # §7; photometric-only, train split only
AUG_COPIES_PER_IMAGE = 2

# Output roots (kept out of git except data.yaml / manifests; see .gitignore).
OUTPUT_BASE_NAME = "Prepared"
YOLO_DATASET_NAME = "MTSD-YOLO"
COCO_DATASET_NAME = "MTSD-COCO"
DATASET_VERSION_TAG = "mtsdv1"        # embedded in manifests and run names

assert abs(sum(SPLIT_RATIOS.values()) - 1.0) < 1e-9, "SPLIT_RATIOS must sum to 1."
print("Configuration loaded.")


## 2. Repository paths

All paths derive from the repository root, located by walking up from the working
directory — the notebook is independent of where Jupyter was launched.


In [ ]:
from __future__ import annotations

import json
import random
import shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "Datasets").exists() and (parent / "Scripts").exists():
            return parent
    raise RuntimeError("Could not locate project root (needs Datasets/ and Scripts/).")


PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "Datasets"
MTSD_ROOT = DATASETS_DIR / "MTSD"
ANNOTATIONS_ROOT = MTSD_ROOT / "Annotations"

PREPARED_ROOT = MTSD_ROOT / OUTPUT_BASE_NAME
YOLO_ROOT = PREPARED_ROOT / YOLO_DATASET_NAME
COCO_ROOT = PREPARED_ROOT / COCO_DATASET_NAME
MANIFEST_PATH = PREPARED_ROOT / "prep_manifest.json"
ARCHIVE_ROOT = PREPARED_ROOT / "_archive"

SPLITS = ("train", "valid", "test")
COCO_ANNOTATION_NAME = "_annotations.coco.json"

print(f"Repository root : {PROJECT_ROOT}")
print(f"Annotations     : {ANNOTATIONS_ROOT}")
print(f"YOLO output     : {YOLO_ROOT}")
print(f"COCO output     : {COCO_ROOT}")


## 3. Discovery and group indexing

Load every QA JSON, verify that all groups share an identical class vocabulary, and build
one record per annotated image (group, resolved image path, dimensions, boxes). The class
map fixes YOLO ids as the QA ids in ascending order, remapped to a contiguous 0-based
range — this ordering is recorded in the manifest and must never change once training has
started.


In [ ]:
qa_files = sorted(ANNOTATIONS_ROOT.glob("GRP-*/Final-QA/QA-GRP*.json"))
if not qa_files:
    raise FileNotFoundError(f"No QA annotation files found under {ANNOTATIONS_ROOT}")

group_payloads = {}
for qa_path in qa_files:
    group = qa_path.parents[1].name          # Annotations/<GRP-n>/Final-QA/QA-*.json
    group_payloads[group] = json.loads(qa_path.read_text(encoding="utf-8-sig"))
    print(f"{group}: {qa_path.relative_to(PROJECT_ROOT)}")

# --- unified class vocabulary --------------------------------------------------
vocabularies = {
    group: tuple(sorted((c["id"], c["name"]) for c in payload.get("categories", [])))
    for group, payload in group_payloads.items()
}
if len(set(vocabularies.values())) != 1:
    raise ValueError(f"Groups disagree on the class vocabulary: {vocabularies}")

qa_categories = sorted(next(iter(vocabularies.values())))       # [(qa_id, name), ...]
CLASS_NAMES = [name for _, name in qa_categories]               # YOLO order (0-based)
QA_ID_TO_CONTIGUOUS = {qa_id: index for index, (qa_id, _) in enumerate(qa_categories)}
print(f"\n{len(CLASS_NAMES)} classes: {CLASS_NAMES}")

# --- per-image records ----------------------------------------------------------
records = []            # one dict per annotated image
for group, payload in sorted(group_payloads.items()):
    boxes_by_image = defaultdict(list)
    for annotation in payload.get("annotations", []):
        boxes_by_image[annotation["image_id"]].append(annotation)
    for image in payload.get("images", []):
        source_rel = str(image.get("source_image", "")).replace("\\", "/")
        records.append({
            "group": group,
            "image_id": image["id"],
            "file_name": image["file_name"],
            "source_path": PROJECT_ROOT / source_rel if source_rel else None,
            "width": image.get("width"),
            "height": image.get("height"),
            "boxes": boxes_by_image.get(image["id"], []),
        })

group_index = pd.DataFrame([
    {
        "group": group,
        "annotated_images": sum(r["group"] == group for r in records),
        "annotations": sum(len(r["boxes"]) for r in records if r["group"] == group),
    }
    for group in sorted(group_payloads)
])
group_index


## 4. Validation of image–annotation pairs

Each record must point at a real, non-empty image file and carry only geometrically valid
boxes. Boxes are clipped to the image bounds; boxes that degenerate to under one pixel of
width or height after clipping are dropped. Nothing is written yet — this is a read-only
audit that feeds the split.


In [ ]:
MIN_BOX_SIDE_PX = 1.0

valid_records = []
missing_images = []
empty_after_validation = []
issue_counts = Counter()

for record in records:
    path = record["source_path"]
    if path is None or not path.exists():
        missing_images.append({"group": record["group"], "file_name": record["file_name"],
                               "expected_path": str(path)})
        issue_counts["missing_image_file"] += 1
        continue

    width, height = record["width"], record["height"]
    clean_boxes = []
    for annotation in record["boxes"]:
        category_id = annotation.get("category_id")
        if category_id not in QA_ID_TO_CONTIGUOUS:
            issue_counts["invalid_class_id"] += 1
            continue
        x, y, bw, bh = annotation.get("bbox", [None] * 4)
        if None in (x, y, bw, bh):
            issue_counts["malformed_bbox"] += 1
            continue
        x0, y0 = max(0.0, float(x)), max(0.0, float(y))
        x1, y1 = min(float(width), float(x) + float(bw)), min(float(height), float(y) + float(bh))
        if x1 - x0 < MIN_BOX_SIDE_PX or y1 - y0 < MIN_BOX_SIDE_PX:
            issue_counts["degenerate_box_after_clip"] += 1
            continue
        if x1 - x0 < float(bw) - 0.5 or y1 - y0 < float(bh) - 0.5:
            issue_counts["box_clipped_to_image"] += 1
        clean_boxes.append({"class_id": QA_ID_TO_CONTIGUOUS[category_id],
                            "x0": x0, "y0": y0, "x1": x1, "y1": y1})

    if not clean_boxes:
        empty_after_validation.append(f"{record['group']}/{record['file_name']}")
        issue_counts["image_without_valid_boxes"] += 1
        # Kept as a negative (background) sample: YOLO treats empty label files as
        # background images; COCO simply lists the image with no annotations.
    record["clean_boxes"] = clean_boxes
    valid_records.append(record)

# Output file names are group-prefixed, which makes cross-group name collisions impossible.
for record in valid_records:
    record["out_name"] = f"{record['group'].lower().replace('-', '')}_{record['file_name']}"

print(f"Usable images          : {len(valid_records):,}")
print(f"Missing image files    : {len(missing_images):,}  (excluded; listed in the manifest)")
print(f"Validation findings    : {dict(issue_counts)}")
pd.DataFrame(missing_images).groupby("group").size().rename("missing_files").to_frame() \
    if missing_images else print("No missing files.")


## 5. Train/validation/test split

`random` mode shuffles each group independently with the fixed seed and cuts it by the
configured ratios, so every split contains images from every group (collection batch is
not confounded with the split). `by-group` mode assigns whole groups per
`BY_GROUP_ASSIGNMENT` instead.


In [ ]:
def assign_splits(records_in, mode, ratios, seed):
    assignment = {}
    if mode == "by-group":
        unknown = {r["group"] for r in records_in} - set(BY_GROUP_ASSIGNMENT)
        if unknown:
            raise ValueError(f"BY_GROUP_ASSIGNMENT is missing groups: {sorted(unknown)}")
        for record in records_in:
            assignment[record["out_name"]] = BY_GROUP_ASSIGNMENT[record["group"]]
        return assignment
    if mode != "random":
        raise ValueError(f"Unknown SPLIT_MODE: {mode!r}")

    rng = random.Random(seed)
    by_group = defaultdict(list)
    for record in records_in:
        by_group[record["group"]].append(record)
    for group, group_records in sorted(by_group.items()):
        group_records = sorted(group_records, key=lambda r: r["out_name"])  # deterministic
        rng.shuffle(group_records)
        n = len(group_records)
        n_train = round(n * ratios["train"])
        n_valid = round(n * ratios["valid"])
        for index, record in enumerate(group_records):
            split = ("train" if index < n_train
                     else "valid" if index < n_train + n_valid
                     else "test")
            assignment[record["out_name"]] = split
    return assignment


split_assignment = assign_splits(valid_records, SPLIT_MODE, SPLIT_RATIOS, RANDOM_SEED)
for record in valid_records:
    record["split"] = split_assignment[record["out_name"]]

split_summary = pd.DataFrame([
    {
        "split": split,
        "images": sum(r["split"] == split for r in valid_records),
        "boxes": sum(len(r["clean_boxes"]) for r in valid_records if r["split"] == split),
        **{group: sum(r["split"] == split and r["group"] == group for r in valid_records)
           for group in sorted(group_payloads)},
    }
    for split in SPLITS
])
display(split_summary)

class_by_split = pd.DataFrame(
    [
        {"class_name": CLASS_NAMES[class_id],
         **{split: sum(1 for r in valid_records if r["split"] == split
                       for b in r["clean_boxes"] if b["class_id"] == class_id)
            for split in SPLITS}}
        for class_id in range(len(CLASS_NAMES))
    ]
)
class_by_split["total"] = class_by_split[list(SPLITS)].sum(axis=1)
class_by_split.sort_values("total", ascending=False)


## 6. Write the YOLO and COCO datasets

Images are hard-linked when the file system allows it (zero extra space) and copied
otherwise; labels are derived from the validated boxes. Existing outputs are never
overwritten silently: with `OVERWRITE_EXISTING = False` the cell refuses to run, with
`True` the previous tree is moved into `Prepared/_archive/<timestamp>/` first.

**This cell writes ~2 GB of data. It is intentionally guarded — review the tables above
before running it.**


In [ ]:
def guard_output_dir(path):
    if not path.exists():
        return
    if not OVERWRITE_EXISTING:
        raise RuntimeError(
            f"{path} already exists. Set OVERWRITE_EXISTING = True to archive and rebuild, "
            f"or point YOLO_DATASET_NAME/COCO_DATASET_NAME at a fresh name."
        )
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    archive_target = ARCHIVE_ROOT / f"{path.name}-{stamp}"
    archive_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(archive_target))
    print(f"Archived previous {path.name} -> {archive_target}")


def place_image(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        import os
        os.link(src, dst)            # hardlink (same volume); zero extra space
    except OSError:
        shutil.copy2(src, dst)       # exFAT/cross-volume fallback


if not REGENERATE_SPLITS and MANIFEST_PATH.exists():
    raise RuntimeError(
        f"{MANIFEST_PATH} exists and REGENERATE_SPLITS is False - nothing to do. "
        f"Set REGENERATE_SPLITS = True to rebuild."
    )

guard_output_dir(YOLO_ROOT)
guard_output_dir(COCO_ROOT)

# --- YOLO layout -----------------------------------------------------------------
for record in valid_records:
    split = record["split"]
    image_dst = YOLO_ROOT / split / "images" / record["out_name"]
    place_image(record["source_path"], image_dst)
    label_lines = []
    for box in record["clean_boxes"]:
        cx = (box["x0"] + box["x1"]) / 2 / record["width"]
        cy = (box["y0"] + box["y1"]) / 2 / record["height"]
        bw = (box["x1"] - box["x0"]) / record["width"]
        bh = (box["y1"] - box["y0"]) / record["height"]
        label_lines.append(f"{box['class_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    label_dst = YOLO_ROOT / split / "labels" / (Path(record["out_name"]).stem + ".txt")
    label_dst.parent.mkdir(parents=True, exist_ok=True)
    label_dst.write_text("\n".join(label_lines) + ("\n" if label_lines else ""),
                         encoding="utf-8")

data_yaml = (
    "train: ../train/images\n"
    "val: ../valid/images\n"
    "test: ../test/images\n"
    "\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
    "\n"
    "mtsd:\n"
    f"  version: {DATASET_VERSION_TAG}\n"
    f"  source: QA-verified MTSD annotations ({', '.join(sorted(group_payloads))})\n"
    f"  split_mode: {SPLIT_MODE}\n"
    f"  seed: {RANDOM_SEED}\n"
)
(YOLO_ROOT / "data.yaml").write_text(data_yaml, encoding="utf-8")
print(f"YOLO dataset written: {YOLO_ROOT}")

# --- COCO layout (RF-DETR) ---------------------------------------------------------
coco_categories = [
    {"id": index, "name": name, "supercategory": "mtsd"}
    for index, name in enumerate(CLASS_NAMES)
]
for split in SPLITS:
    split_records = [r for r in valid_records if r["split"] == split]
    images_json, annotations_json = [], []
    annotation_id = 1
    for image_id, record in enumerate(split_records, start=1):
        place_image(record["source_path"], COCO_ROOT / split / record["out_name"])
        images_json.append({
            "id": image_id, "file_name": record["out_name"],
            "width": record["width"], "height": record["height"],
            "license": 0, "date_captured": None,
            "mtsd_group": record["group"], "mtsd_source": record["file_name"],
        })
        for box in record["clean_boxes"]:
            bw, bh = box["x1"] - box["x0"], box["y1"] - box["y0"]
            annotations_json.append({
                "id": annotation_id, "image_id": image_id,
                "category_id": box["class_id"],
                "bbox": [round(box["x0"], 2), round(box["y0"], 2), round(bw, 2), round(bh, 2)],
                "area": round(bw * bh, 2), "iscrowd": 0, "segmentation": [],
            })
            annotation_id += 1
    coco_payload = {
        "info": {"description": f"MTSD detection dataset ({DATASET_VERSION_TAG}), "
                                f"split={split}, prepared from QA-verified annotations",
                 "date_created": datetime.now().isoformat(timespec="seconds")},
        "licenses": [], "categories": coco_categories,
        "images": images_json, "annotations": annotations_json,
    }
    annotation_path = COCO_ROOT / split / COCO_ANNOTATION_NAME
    annotation_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_path.write_text(json.dumps(coco_payload, indent=2), encoding="utf-8")
    print(f"COCO {split}: {len(images_json):,} images, {len(annotations_json):,} annotations")
print(f"COCO dataset written: {COCO_ROOT}")


## 7. Optional offline augmentation — disabled by default

Photometric variants (brightness, contrast, mild Gaussian blur) of **training images
only**; the label files are reused unchanged because no geometry is touched. Geometric
flips are deliberately excluded: traffic signs are not mirror-invariant (text, arrows,
and pole-side geometry all flip). Enable via `ENABLE_OFFLINE_AUGMENTATION = True` in §1.

**Augmentation cell — intentionally left unexecuted during repository setup.**


In [ ]:
if not ENABLE_OFFLINE_AUGMENTATION:
    print("Offline augmentation is disabled (ENABLE_OFFLINE_AUGMENTATION = False).")
else:
    from PIL import Image, ImageEnhance, ImageFilter

    rng = random.Random(RANDOM_SEED)
    train_images_dir = YOLO_ROOT / "train" / "images"
    train_labels_dir = YOLO_ROOT / "train" / "labels"
    originals = sorted(p for p in train_images_dir.iterdir()
                       if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
                       and "_aug" not in p.stem)
    created = 0
    for image_path in originals:
        label_path = train_labels_dir / (image_path.stem + ".txt")
        for copy_index in range(1, AUG_COPIES_PER_IMAGE + 1):
            out_stem = f"{image_path.stem}_aug{copy_index}"
            out_image = train_images_dir / f"{out_stem}{image_path.suffix}"
            if out_image.exists():
                continue
            with Image.open(image_path) as img:
                augmented = img.convert("RGB")
                augmented = ImageEnhance.Brightness(augmented).enhance(rng.uniform(0.8, 1.2))
                augmented = ImageEnhance.Contrast(augmented).enhance(rng.uniform(0.85, 1.15))
                if rng.random() < 0.5:
                    augmented = augmented.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, 1.0)))
                augmented.save(out_image, quality=92)
            if label_path.exists():
                (train_labels_dir / f"{out_stem}.txt").write_text(
                    label_path.read_text(encoding="utf-8"), encoding="utf-8")
            created += 1
    print(f"Created {created} augmented training images "
          f"({AUG_COPIES_PER_IMAGE} photometric copies per original).")
    print("Note: rerun §8 checks and consider a distinct DATASET_VERSION_TAG "
          "(e.g. 'mtsdv1-aug') so runs remain distinguishable.")


## 8. Post-preparation integrity checks

Re-read what was actually written to disk (rather than trusting in-memory state):
image/label pairing, empty and malformed labels, per-split class distribution, and
YOLO-vs-COCO consistency.


In [ ]:
check_rows, label_class_counts = [], defaultdict(Counter)
problems = Counter()

for split in SPLITS:
    images_dir = YOLO_ROOT / split / "images"
    labels_dir = YOLO_ROOT / split / "labels"
    image_stems = {p.stem for p in images_dir.iterdir() if p.is_file()} if images_dir.exists() else set()
    label_stems = {p.stem for p in labels_dir.glob("*.txt")} if labels_dir.exists() else set()

    empty_labels = invalid_rows = 0
    for label_path in (labels_dir.glob("*.txt") if labels_dir.exists() else []):
        lines = [l for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        if not lines:
            empty_labels += 1
        for line in lines:
            parts = line.split()
            try:
                class_id = int(parts[0])
                values = [float(v) for v in parts[1:5]]
                assert len(parts) == 5 and 0 <= class_id < len(CLASS_NAMES)
                assert all(0.0 <= v <= 1.0 for v in values) and values[2] > 0 and values[3] > 0
                label_class_counts[split][CLASS_NAMES[class_id]] += 1
            except (AssertionError, IndexError, ValueError):
                invalid_rows += 1

    coco_path = COCO_ROOT / split / COCO_ANNOTATION_NAME
    coco_payload = json.loads(coco_path.read_text(encoding="utf-8")) if coco_path.exists() else {}
    problems[split] += len(image_stems - label_stems) + len(label_stems - image_stems) + invalid_rows
    check_rows.append({
        "split": split,
        "yolo_images": len(image_stems),
        "yolo_labels": len(label_stems),
        "images_without_label": len(image_stems - label_stems),
        "labels_without_image": len(label_stems - image_stems),
        "empty_labels(background)": empty_labels,
        "invalid_label_rows": invalid_rows,
        "coco_images": len(coco_payload.get("images", [])),
        "coco_annotations": len(coco_payload.get("annotations", [])),
    })

integrity_table = pd.DataFrame(check_rows)
display(integrity_table)

class_distribution_table = pd.DataFrame(
    [{"class_name": name, **{split: label_class_counts[split].get(name, 0) for split in SPLITS}}
     for name in CLASS_NAMES]
)
class_distribution_table["total"] = class_distribution_table[list(SPLITS)].sum(axis=1)
display(class_distribution_table.sort_values("total", ascending=False))

if sum(problems.values()):
    print(f"PROBLEMS FOUND: {dict(problems)} - inspect before training.")
else:
    print("All integrity checks passed.")


## 9. Summary report

The manifest is the provenance record for every training run built on this dataset: it
pins the configuration, the class order, the split assignment counts, and the exclusion
list, and is what the training notebooks cite as `dataset_source`.


In [ ]:
manifest = {
    "dataset_version": DATASET_VERSION_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_annotations": [str(p.relative_to(PROJECT_ROOT)) for p in qa_files],
    "configuration": {
        "split_mode": SPLIT_MODE,
        "split_ratios": SPLIT_RATIOS,
        "by_group_assignment": BY_GROUP_ASSIGNMENT if SPLIT_MODE == "by-group" else None,
        "random_seed": RANDOM_SEED,
        "offline_augmentation": ENABLE_OFFLINE_AUGMENTATION,
        "aug_copies_per_image": AUG_COPIES_PER_IMAGE if ENABLE_OFFLINE_AUGMENTATION else 0,
        "min_box_side_px": MIN_BOX_SIDE_PX,
    },
    "class_names": CLASS_NAMES,
    "qa_id_to_contiguous": {str(k): v for k, v in QA_ID_TO_CONTIGUOUS.items()},
    "outputs": {"yolo": str(YOLO_ROOT.relative_to(PROJECT_ROOT)),
                "coco": str(COCO_ROOT.relative_to(PROJECT_ROOT))},
    "split_summary": split_summary.to_dict(orient="records"),
    "class_distribution_by_split": class_distribution_table.to_dict(orient="records"),
    "validation_findings": dict(issue_counts),
    "missing_images_excluded": missing_images,
    "images_kept_as_background": empty_after_validation,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str) + "\n", encoding="utf-8")

print(f"Manifest written: {MANIFEST_PATH}")
print(f"\nPrepared dataset ready for:")
print(f"  YOLO12.ipynb / YOLO26.ipynb -> {YOLO_ROOT.relative_to(PROJECT_ROOT)}")
print(f"  RF-DETR.ipynb               -> {COCO_ROOT.relative_to(PROJECT_ROOT)}")
